In [1]:
import os
import pandas as pd
from datetime import datetime
import pytz
import unicodedata

# === Date & Paths ===
eastern = pytz.timezone("US/Eastern")
today = datetime.now(eastern).date()
today_str = today.isoformat()

paths = {
    "spreads": f"../data/novig-odds/novig_spreads_{today_str}.csv",
    "pitching_stats": f"../data/pitching_stats/pitching_stats_{today_str}.csv",
    "standings": f"../data/standings/standings_{today_str}.csv",
    "team_batting": f"../data/team_batting/team_batting_{today_str}.csv",
    "team_pitching": f"../data/team_pitching/team_pitching_{today_str}.csv"
}

# === Load Data ===
for label, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

spreads_df = pd.read_csv(paths["spreads"])
pitching_stats_df = pd.read_csv(paths["pitching_stats"])
standings_df = pd.read_csv(paths["standings"])
batting_df = pd.read_csv(paths["team_batting"])
teamp_df = pd.read_csv(paths["team_pitching"])

# === Normalize ===
def normalize_str(s):
    if not isinstance(s, str): return ""
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8').strip().lower()

# === Preprocess Spreads ===
spreads_df['fav_score'] = None
spreads_df['dog_score'] = None
spreads_df['home_score'] = None
spreads_df['away_score'] = None

# === Normalize team names ===
pitching_stats_df['Home Team Clean'] = pitching_stats_df['Home Team'].apply(normalize_str)
pitching_stats_df['Away Team Clean'] = pitching_stats_df['Away Team'].apply(normalize_str)

# === Map team abbreviations to full names ===
team_name_map = {
    "ATL": "Atlanta Braves", "PHI": "Philadelphia Phillies", "OAK": "Athletics",
    "TOR": "Toronto Blue Jays", "TB": "Tampa Bay Rays", "HOU": "Houston Astros",
    "WAS": "Washington Nationals", "SEA": "Seattle Mariners", "CIN": "Cincinnati Reds",
    "CHC": "Chicago Cubs", "MIL": "Milwaukee Brewers", "CWS": "Chicago White Sox",
    "BAL": "Baltimore Orioles", "LAA": "Los Angeles Angels", "CLE": "Cleveland Guardians",
    "COL": "Colorado Rockies", "NYM": "New York Mets", "SF": "San Francisco Giants",
    "MIA": "Miami Marlins", "BOS": "Boston Red Sox", "STL": "St. Louis Cardinals",
    "TEX": "Texas Rangers", "DET": "Detroit Tigers", "KC": "Kansas City Royals",
    "KAN": "Kansas City Royals", "AZ": "Arizona Diamondbacks", "ARI": "Arizona Diamondbacks",
    "PIT": "Pittsburgh Pirates", "SD": "San Diego Padres", "MIN": "Minnesota Twins",
    "NYY": "New York Yankees", "LAD": "Los Angeles Dodgers"
}

# Derive home and away teams
spreads_df['home_team'] = spreads_df.apply(lambda row: row['fav_team'] if row['home_favorite'] == 1 else row['dog_team'], axis=1)
spreads_df['away_team'] = spreads_df.apply(lambda row: row['dog_team'] if row['home_favorite'] == 1 else row['fav_team'], axis=1)
spreads_df['home_team_full'] = spreads_df['home_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['away_team_full'] = spreads_df['away_team'].map(lambda abbr: team_name_map.get(abbr.upper(), ""))
spreads_df['home_team_clean'] = spreads_df['home_team_full'].apply(normalize_str)
spreads_df['away_team_clean'] = spreads_df['away_team_full'].apply(normalize_str)

# === Assign Pitchers and Stats ===
output_rows = []

for _, row in spreads_df.iterrows():
    home = row['home_team_clean']
    away = row['away_team_clean']

    ps_row = pitching_stats_df[
        (pitching_stats_df['Home Team Clean'] == home) &
        (pitching_stats_df['Away Team Clean'] == away)
    ]

    if ps_row.empty:
        ps_row = pitching_stats_df[
            (pitching_stats_df['Home Team Clean'] == away) &
            (pitching_stats_df['Away Team Clean'] == home)
        ]
        flipped = not ps_row.empty
    else:
        flipped = False

    if not ps_row.empty:
        ps_row = ps_row.iloc[0]
        if flipped:
            home_pitcher, away_pitcher = ps_row['Away Starter'], ps_row['Home Starter']
            home_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }
            away_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
        else:
            home_pitcher, away_pitcher = ps_row['Home Starter'], ps_row['Away Starter']
            home_stats = {
                'era': ps_row['Home ERA'], 'whip': ps_row['Home WHIP'], 'so': ps_row['Home SO'],
                'ip': ps_row['Home IP'], 'avg': ps_row['Home AVG']
            }
            away_stats = {
                'era': ps_row['Away ERA'], 'whip': ps_row['Away WHIP'], 'so': ps_row['Away SO'],
                'ip': ps_row['Away IP'], 'avg': ps_row['Away AVG']
            }

        if row['home_favorite'] == 1:
            fav_pitcher, dog_pitcher = home_pitcher, away_pitcher
            fav_stats, dog_stats = home_stats, away_stats
        else:
            fav_pitcher, dog_pitcher = away_pitcher, home_pitcher
            fav_stats, dog_stats = away_stats, home_stats
    else:
        fav_pitcher = dog_pitcher = "Unknown"
        fav_stats = dog_stats = {k: None for k in ['era', 'whip', 'so', 'ip', 'avg']}

    row = row.copy()
    row['fav_pitcher'] = fav_pitcher
    row['dog_pitcher'] = dog_pitcher
    for k in fav_stats:
        row[f'fav_pitcher_{k}'] = fav_stats[k]
        row[f'dog_pitcher_{k}'] = dog_stats[k]

    output_rows.append(row)

merged_df = pd.DataFrame(output_rows)

# === Normalize and Join Team Stats ===
for df in [standings_df, batting_df, teamp_df]:
    df['team_clean'] = df['Tm'].apply(normalize_str)

def get_team_stat(team_abbr, df, column):
    full = team_name_map.get(team_abbr.upper(), "")
    clean = normalize_str(full)
    row = df[df['team_clean'] == clean]
    return row[column].values[0] if not row.empty else None

# Standings
merged_df['fav_win_pct'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))
merged_df['dog_win_pct'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, 'W-L%'))

# Batting/ERA
merged_df['fav_ba'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['dog_ba'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, 'BA'))
merged_df['fav_era_p'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))
merged_df['dog_era_p'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, 'ERA'))

# Additional standings columns
standings_columns = ['W','L','Strk','R','RA','SOS','SRS','pythWL','Luck','last10','last20','last30']
for col in standings_columns:
    merged_df[f'fav_{col}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))
    merged_df[f'dog_{col}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, standings_df, col))

# Additional batting columns
batting_columns = ['R/G','PA','AB','R','H','2B','3B','HR','RBI','SB','BB','SO','BA','OBP','SLG','OPS','OPS+']
for col in batting_columns:
    col_renamed = f'{col}_b' if col in ['R','H','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, batting_df, col))

# Additional pitching columns
pitching_columns = ['RA/G','ERA','SV','H','R','ER','HR','BB','SO','ERA+','FIP','WHIP','H9']
for col in pitching_columns:
    col_renamed = f'{col}_p' if col in ['H','R','BB','SO'] else col
    merged_df[f'fav_{col_renamed}'] = merged_df['fav_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))
    merged_df[f'dog_{col_renamed}'] = merged_df['dog_team'].apply(lambda abbr: get_team_stat(abbr, teamp_df, col))

# === Final Ordering ===
first_cols = [
    'game_time_est', 'fav_team', 'fav_score', 'dog_team', 'dog_score',
    'home_team', 'home_score', 'away_team', 'away_score',
    'fav_line', 'dog_line', 'fav_price', 'dog_price',
    'fav_price_american', 'dog_price_american', 'home_favorite', 'market',
    'fav_pitcher', 'dog_pitcher',
    'fav_pitcher_era', 'dog_pitcher_era',
    'fav_pitcher_whip', 'dog_pitcher_whip',
    'fav_pitcher_so', 'dog_pitcher_so',
    'fav_pitcher_ip', 'dog_pitcher_ip',
    'fav_pitcher_avg', 'dog_pitcher_avg',
    'fav_win_pct', 'dog_win_pct', 'fav_ba', 'dog_ba', 'fav_era_p', 'dog_era_p'
]
remaining_cols = [col for col in merged_df.columns if col not in first_cols and col not in ['market_timestamp', 'home_team_full', 'away_team_full', 'home_team_clean', 'away_team_clean']]
merged_df = merged_df[first_cols + remaining_cols]

# === Output ===
print("\u2705 Final enriched training set (WAR excluded):")
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
display(merged_df)

# === Save CSV ===
output_path = f"../training-data/training-set/training_set_{today_str}.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_df.to_csv(output_path, index=False)
print(f"\u2705 Enriched training set saved to {output_path}")


✅ Final enriched training set (WAR excluded):


,game_time_est,fav_team,fav_score,dog_team,dog_score,home_team,home_score,away_team,away_score,fav_line,dog_line,fav_price,dog_price,fav_price_american,dog_price_american,home_favorite,market,fav_pitcher,dog_pitcher,fav_pitcher_era,dog_pitcher_era,fav_pitcher_whip,dog_pitcher_whip,fav_pitcher_so,dog_pitcher_so,fav_pitcher_ip,dog_pitcher_ip,fav_pitcher_avg,dog_pitcher_avg,fav_win_pct,dog_win_pct,fav_ba,dog_ba,fav_era_p,dog_era_p,fav_W,dog_W,fav_L,dog_L,fav_Strk,dog_Strk,fav_R,dog_R,fav_RA,dog_RA,fav_SOS,dog_SOS,fav_SRS,dog_SRS,fav_pythWL,dog_pythWL,fav_Luck,dog_Luck,fav_last10,dog_last10,fav_last20,dog_last20,fav_last30,dog_last30,fav_R/G,dog_R/G,fav_PA,dog_PA,fav_AB,dog_AB,fav_R_b,dog_R_b,fav_H_b,dog_H_b,fav_2B,dog_2B,fav_3B,dog_3B,fav_HR,dog_HR,fav_RBI,dog_RBI,fav_SB,dog_SB,fav_BB_b,dog_BB_b,fav_SO_b,dog_SO_b,fav_BA,dog_BA,fav_OBP,dog_OBP,fav_SLG,dog_SLG,fav_OPS,dog_OPS,fav_OPS+,dog_OPS+,fav_RA/G,dog_RA/G,fav_ERA,dog_ERA,fav_SV,dog_SV,fav_H_p,dog_H_p,fav_R_p,dog_R_p,fav_ER,dog_ER,fav_BB_p,dog_BB_p,fav_SO_p,dog_SO_p,fav_ERA+,dog_ERA+,fav_FIP,dog_FIP,fav_WHIP,dog_WHIP,fav_H9,dog_H9
0,2025-06-03T19:40:00-04:00,DET,None,CWS,None,CWS,None,DET,None,-1.5,1.5,0.502,0.522,-101,-109,0,CWS +1.5,TBD,Shane Smith,NaN,2.68,NaN,1.16,NaN,54.0,NaN,57.0,NaN,0.204,0.656,0.300,.251,.220,3.13,4.25,40,18,21,42,W 2,L 4,5.1,3.4,3.5,4.4,-0.1,0.0,1.5,-1.1,41-20,23-37,-1.0,-5.0,7-3,3-7,14-6,7-13,21-9,11-19,5.07,3.37,2315,2185,2065,1955,309,202,519,430,94,89,10,3,57,66,295,195,21,46,202,192,547,504,.251,.220,.324,.292,.413,.341,.736,.633,109,80,3.46,4.43,3.13,4.25,19,5,454,487,211,266,189,243,171,222,521,432,128,97,3.62,4.59,1.151,1.378,7.5,8.5
1,2025-06-03T18:40:00-04:00,MIA,None,COL,None,MIA,None,COL,None,-1.5,1.5,0.448,0.582,123,-139,1,MIA -1.5,Sandy Alcantara,TBD,8.47,NaN,1.67,NaN,40.0,NaN,51.0,NaN,0.279,NaN,0.397,0.167,.249,.217,5.11,5.55,23,10,35,50,L 2,W 1,4.1,3.2,5.5,6.2,0.2,0.4,-1.2,-2.7,21-37,14-46,2.0,-4.0,4-6,2-8,8-12,3-17,11-19,5-25,4.07,3.17,2201,2180,1978,1982,236,190,492,430,98,89,8,15,71,75,226,188,46,28,182,161,483,586,.249,.217,.315,.281,.384,.356,.698,.636,91,71,5.47,6.22,5.11,5.55,11,8,529,614,317,373,293,320,219,206,463,397,86,83,4.47,4.71,1.450,1.579,9.2,10.6
2,2025-06-03T19:07:00-04:00,PHI,None,TOR,None,TOR,None,PHI,None,-1.5,1.5,0.455,0.567,120,-131,0,TOR +1.5,Cristopher Sánchez,Bowden Francis,3.32,5.04,1.31,1.36,70.0,46.0,59.2,55.1,0.246,0.271,0.610,0.525,.256,.254,4.00,4.03,36,31,23,28,L 4,W 5,4.8,4.2,4.4,4.3,-0.2,0.0,0.1,-0.1,32-27,29-30,4.0,2.0,5-5,6-4,13-7,12-8,20-10,18-12,4.76,4.20,2274,2244,2016,1983,281,248,517,504,92,102,9,2,60,81,267,238,56,35,217,205,463,413,.256,.254,.333,.328,.405,.395,.738,.723,104,103,4.39,4.29,4.00,4.03,20,18,515,466,259,253,236,236,181,171,571,537,104,104,3.52,4.16,1.312,1.209,8.7,8.0
3,2025-06-03T18:40:00-04:00,PIT,None,HOU,None,PIT,None,HOU,None,-1.5,1.5,0.396,0.610,153,-156,1,PIT -1.5,Paul Skenes,Lance McCullers Jr.,2.15,5.89,0.92,1.69,77.0,26.0,75.1,18.1,0.187,0.260,0.367,0.542,.227,.254,3.96,3.66,22,32,38,27,L 1,W 1,3.2,4.1,4.2,3.8,0.3,0.2,-0.7,0.4,23-37,31-28,-1.0,1.0,5-5,7-3,9-11,12-8,11-19,16-14,3.23,4.05,2243,2182,1997,1959,194,239,454,497,75,80,10,6,48,59,188,228,58,29,208,172,524,449,.227,.254,.306,.321,.340,.391,.645,.712,81,100,4.20,3.80,3.96,3.66,12,16,473,423,252,224,234,211,190,185,436,560,106,111,3.82,3.58,1.246,1.171,8.0,7.3
4,2025-06-03T19:10:00-04:00,CIN,None,MIL,None,CIN,None,MIL,None,-1.5,1.5,0.364,0.659,175,-193,1,CIN -1.5,Hunter Greene,Freddy Peralta,2.63,2.77,0.91,1.17,66.0,66.0,54.2,65.0,0.195,0.208,0.475,0.541,.244,.238,3.74,3.95,29,33,32,28,L 3,W 8,4.5,4.5,4.0,4.2,-0.1,0.0,0.3,0.3,34-27,32-29,-5.0,1.0,4-6,8-2,9-11,13-7,13-17,17-13,4.51,4.54,2294,2307,2050,2042,275,277,501,487,102,87,7,5,67,64,263,254,55,81,208,211,543,496,.244,.238,.318,.315,.392,.365,.710,.681,92,92,4.03,4.23,3.74,3.95,16,15,463,492,246,258,224,237,184,220,485,495,119,104,4.05,4.12,1.200,1.318,7.7,8.2
5,2025-06-03T19:05:00-04:00,NYY,None,CLE,None,CLE,None,NYY,None,-1.5,1.5,0.488,0.545,105,-120,0,NYY -

✅ Enriched training set saved to ../training-data/training-set/training_set_2025-06-03.csv
